Varshini Narayanan

This code combines the stress_days.csv and firm_list.csv as filters to extract the needed data. The file produced is the complete and cleaned data for this project.

115 lines of code

In [35]:
#%pip install wrds


In [5]:
import wrds
import pandas as pd
import numpy as np

db = wrds.Connection()


WRDS recommends setting up a .pgpass file.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [24]:
#input files
firms = pd.read_csv('firm_list.csv')
stress_days = pd.read_csv('stress_days.csv')

firms_list = tuple(firms['Symbol'].unique())


In [25]:
# Clean up tables
firms['ticker'] = (
    firms['Symbol']
    .str.upper()
    .str.replace('-', '.', regex=False)   # make share-class format match CRSP (e.g., BRK.A -> BRK-A)
)

stress_days['date'] = pd.to_datetime(stress_days['date'])
stress_days['quarter'] = stress_days['date'].dt.to_period('Q')

# Identify stress days and quarters
stress_dates = set(stress_days['date'])
stress_quarters = stress_days['quarter'].astype(str).unique()

# Build ticker list for WRDS query
tickers = tuple(firms['ticker'].dropna().unique())

# Format stress dates for SQL queries
dates = tuple(stress_days['date'].dt.strftime('%Y-%m-%d'))

In [26]:
print(tickers)

('NVDA', 'AAPL', 'GOOG', 'MSFT', 'AMZN', 'TSM', 'META', 'AVGO', 'TSLA', 'BRK.A', 'JPM', 'WMT', 'LLY', 'XOM', 'V', 'JNJ', 'ASML', 'BAC', 'MU', 'MA')


In [27]:
#need to match BRK.A to crsp
brk_tickers = db.raw_sql(
    "SELECT DISTINCT ticker FROM crsp.dsenames WHERE LEFT(ticker,3) = 'BRK'"
)

print(brk_tickers)

   ticker
0     BRK
1    BRKB
2    BRKC
3    BRKD
4    BRKE
5    BRKH
6    BRKL
7   BRKNY
8    BRKR
9    BRKS
10   BRKT
11   BRKU
12   BRKY


In [28]:

# Convert Berkshire tickers to CRSP format
firms.loc[firms['ticker'].str.contains('BRK', na=False), 'ticker'] = 'BRK'

In [29]:
tickers = tuple(firms['ticker'].dropna().unique())
print(tickers)

('NVDA', 'AAPL', 'GOOG', 'MSFT', 'AMZN', 'TSM', 'META', 'AVGO', 'TSLA', 'BRK', 'JPM', 'WMT', 'LLY', 'XOM', 'V', 'JNJ', 'ASML', 'BAC', 'MU', 'MA')


In [30]:
crsp = db.raw_sql(f"""
    SELECT a.date,
           a.permno,
           b.ticker,
           b.ncusip,
           ABS(a.prc) AS prc,
           a.ret,
           a.shrout
    FROM crsp.dsf a
    JOIN crsp.dsenames b
        ON a.permno = b.permno
        AND b.namedt <= a.date
        AND a.date <= b.nameendt
    WHERE b.ticker IN {tickers}
""")

crsp['date'] = pd.to_datetime(crsp['date'])

# keep just the stress quarters
crsp['quarter'] = crsp['date'].dt.to_period('Q').astype(str)
crsp = crsp[crsp['quarter'].isin(stress_quarters)]

In [31]:
#stress day binary variable
crsp['stress_day'] = crsp['date'].isin(stress_dates).astype(int)

In [32]:
#map stress dates to most recent quarter end date
crsp['fdate'] = (
    crsp['date']
    .dt.to_period('Q')
    .dt.to_timestamp('Q')
    - pd.offsets.QuarterEnd(1)
)

In [33]:
#daily returns
crsp['daily_returns'] = crsp['ret']

In [34]:
market = db.raw_sql("""
    SELECT date, vwretd
    FROM crsp.dsi
""")

market['date'] = pd.to_datetime(market['date'])
market['market_volatility'] = market['vwretd'].abs()

crsp = crsp.merge(
    market[['date','market_volatility']],
    on='date',
    how='left'
)

In [35]:
#linking permno (stock identifier) to GVKEY
link = db.raw_sql("""
    SELECT gvkey, lpermno as permno
    FROM crsp.ccmxpf_linktable
    WHERE linktype IN ('LU','LC')
    AND usedflag = 1
""")

crsp = crsp.merge(link, on='permno', how='left')

In [36]:
#get quarters and cuisps
needed_quarters = tuple(crsp['fdate'].dt.strftime('%Y-%m-%d').unique())
needed_cusips = tuple(crsp['ncusip'].dropna().unique())

In [37]:
#pull 13F
inst = db.raw_sql(f"""
    SELECT fdate, cusip, mgrno, shares
    FROM tfn.s34
    WHERE fdate IN {needed_quarters}
    AND cusip IN {needed_cusips}
""")

inst['fdate'] = pd.to_datetime(inst['fdate'])

In [38]:
#merge institutional data
df = crsp.merge(
    inst,
    left_on=['ncusip','fdate'],
    right_on=['cusip','fdate'],
    how='left'
)

In [39]:
#merge market volatility data into df
# ensure dates match format
df['date'] = pd.to_datetime(df['date'])
market['date'] = pd.to_datetime(market['date'])


In [40]:
#ownership percentage
df['ownership_pct'] = df['shares'] / (df['shrout'] * 1000)
df['shrout'] = (df['shrout'] * 1000) #scale so shares outstanding are not in the 1000s anymore

In [41]:
df.columns

Index(['date', 'permno', 'ticker', 'ncusip', 'prc', 'ret', 'shrout', 'quarter',
       'stress_day', 'fdate', 'daily_returns', 'market_volatility', 'gvkey',
       'cusip', 'mgrno', 'shares', 'ownership_pct'],
      dtype='object')

In [42]:
print(df.ticker.unique())

<StringArray>
['MSFT',  'XOM', 'META', 'GOOG', 'AAPL',  'BRK',  'JNJ',  'JPM',  'LLY',
   'MU',  'WMT', 'ASML',  'BAC',  'TSM', 'NVDA', 'AMZN',   'MA', 'AVGO',
 'TSLA',    'V']
Length: 20, dtype: string


In [43]:
final_df = df[[
    'ticker',
    'date',
    'daily_returns',
    'market_volatility',
    'mgrno',
    'ownership_pct',
    'shares',
    'shrout',
    'stress_day'
]]

final_df = final_df.sort_values(
    by=['ticker', 'date', 'ownership_pct'],
    ascending=[True, True, False]
)


In [44]:
final_df.head

<bound method NDFrame.head of         ticker       date  daily_returns  market_volatility    mgrno  \
8677666   AAPL 2018-01-02       0.017905           0.008505  90457.0   
8675337   AAPL 2018-01-02       0.017905           0.008505   9385.0   
8677606   AAPL 2018-01-02       0.017905           0.008505  81540.0   
8675041   AAPL 2018-01-02       0.017905           0.008505   8350.0   
8677164   AAPL 2018-01-02       0.017905           0.008505  27800.0   
...        ...        ...            ...                ...      ...   
6304813    XOM 2022-12-30       0.010073           0.002451   9628.0   
6301631    XOM 2022-12-30       0.010073           0.002451   9666.0   
6302641    XOM 2022-12-30       0.010073           0.002451  14648.0   
6304824    XOM 2022-12-30       0.010073           0.002451   9666.0   
6305834    XOM 2022-12-30       0.010073           0.002451  14648.0   

         ownership_pct       shares        shrout  stress_day  
8677666       0.068501  348468032.0  5087

In [45]:
#check if stress day calculated properly
crsp['stress_day'].mean()

np.float64(0.3277834341498473)

In [46]:
final_df.isna().sum()

ticker                0
date                  0
daily_returns         0
market_volatility     0
mgrno                61
ownership_pct        61
shares               61
shrout                0
stress_day            0
dtype: int64

In [47]:
#see where N/As are happening
df[df['ownership_pct'].isna()][['ticker','date','mgrno','shares']]

,ticker,date,mgrno,shares
34465258,AVGO,2018-01-02,<NA>,<NA>
34465259,AVGO,2018-01-03,<NA>,<NA>
34465260,AVGO,2018-01-04,<NA>,<NA>
34465261,AVGO,2018-01-05,<NA>,<NA>
34465262,AVGO,2018-01-08,<NA>,<NA>
...,...,...,...,...
34465314,AVGO,2018-03-23,<NA>,<NA>
34465315,AVGO,2018-03-26,<NA>,<NA>
34465316,AVGO,2018-03-27,<NA>,<NA>
34465317,AVGO,2018-03-28,<NA>,<NA>


Y0982710 is the code for a foreign domiciled institution, so data is not available from the dataset.

In [48]:
df['ownership_pct'].isna().mean()

np.float64(1.5840802531412182e-06)

In [49]:
#filter out small holdings (<.1% cutoff), including Y0982710

df = df[df['ownership_pct'] >= 0.001]

In [109]:
print(df.ticker.unique())

<StringArray>
['MSFT',  'XOM', 'META', 'GOOG',  'BRK', 'AAPL',  'JNJ',  'LLY',   'MU',
  'JPM',  'WMT',  'BAC', 'AMZN', 'ASML',  'TSM', 'NVDA',   'MA', 'AVGO',
    'V', 'TSLA']
Length: 20, dtype: string


In [50]:
final_df.to_csv('str_vol_inst_data.csv', index=False)